# Comparing p53 DNA-damage response modelling and expression-based signatures for breast cancer doxorubicin response and survival

## Abstract

This assignment compared mechanistic p53 DNA-damage response modelling with expression-based regression signatures in breast cancer. TCGA-BRCA tumour expression and clinical follow-up were used for patient survival analyses, while DepMap/CCLE expression and PRISM doxorubicin viability response were used for breast cancer cell-line drug-response analyses. A simplified p53 ODE score from the given course template showed a modest continuous association with TCGA-BRCA overall survival, but did not predict PRISM doxorubicin sensitivity. Expression-based transfer models also showed weak cross-domain transfer: a PRISM-derived doxorubicin signature did not predict TCGA survival, and a TCGA-derived survival score did not predict PRISM doxorubicin response. Overall, the strongest results were within-domain, especially for TCGA survival, showing that patient prognosis and in vitro drug sensitivity are related but non-equivalent endpoints.

## Submission Note

This notebook was used to generate the final PDF report. Supporting notebooks contain the detailed data preparation and modelling steps.

## Introduction

Breast cancer is a common and clinically heterogeneous cancer. Doxorubicin is an anthracycline chemotherapy used in breast cancer treatment, and its activity is closely linked to DNA damage, replication stress and cell-death pathways. This makes doxorubicin a useful example for a personalised medicine assignment: the biological target process is plausible, but measured patient benefit depends on many factors beyond tumour-cell intrinsic drug sensitivity.

The p53 pathway is central to the DNA-damage response. After DNA damage, p53 can promote cell-cycle arrest, DNA repair, senescence or apoptosis depending on cellular context and signal strength. Phosphorylation markers such as p53 S15 and p53 S46 are often interpreted as different aspects of p53 activation. This project therefore asks whether p53/DNA-damage response features, simulated p53 ODE scores and expression-based signatures can help explain either patient survival or doxorubicin response.

The main personalised medicine issue is endpoint transfer. TCGA-BRCA provides patient tumour expression and survival [1], but not direct doxorubicin response. PRISM provides in vitro doxorubicin viability response [3], but not patient outcomes. Comparing these sources tests whether a biologically motivated signal transfers between cell-line drug sensitivity and patient prognosis.

## Assignment Questions and Study Design

The analysis was organised around six questions:

1. **Q1:** Are p53 ODE response scores associated with TCGA-BRCA patient survival?
2. **Q2:** Are p53 ODE response scores associated with PRISM doxorubicin sensitivity in breast cancer cell lines?
3. **Q3:** Can a PRISM-derived doxorubicin expression signature be transferred to TCGA-BRCA survival?
4. **Q4:** Can a TCGA-derived survival expression score be transferred to PRISM doxorubicin sensitivity?
5. **Q5:** How do mechanistic p53 ODE scores compare with ML/regression-based approaches?
6. **Q6:** Is a LINCS doxorubicin-induced expression signature available to support the p53/DNA-damage interpretation?

The data sources answer different parts of the assignment. TCGA-BRCA is used for patient tumour expression and survival [1]. DepMap/CCLE is used for baseline breast cancer cell-line expression [2]. PRISM is used for doxorubicin viability response in cell lines [3]. LINCS would be used for drug-induced gene-expression changes if suitable doxorubicin perturbation data were available [4]. The given p53 ODE/course template is used to simulate DNA-damage response scores, especially p53 S15 and p53 S46 summaries across DDR inputs.

## Data and Preprocessing

The validated TCGA-BRCA table contained **1,094 patient-level rows**, **151 deaths/events** and **943 censored patients** [1]. Age and stage were available as clinical covariates, and all 16 shared p53/DNA-damage genes were present: ATM, CHEK2, HIPK2, MDM2, PPM1D, SIAH1, TP53, WSB1, CDKN1A, BAX, BBC3, GADD45A, MDM4, ATR, CHEK1 and CASP3. The Q1, Q3 and Q4 survival model tables used **1,081 complete model rows** after joining the relevant scores.

The validated PRISM/DepMap table contained **26 breast cancer cell lines** with baseline DepMap/CCLE expression and non-missing PRISM doxorubicin response [2,3]. PRISM doxorubicin log-fold-change was interpreted as relative abundance or viability versus control, not as gene-expression change. A more negative PRISM LFC means lower relative abundance after treatment, so the analysis defined a convenience sensitivity score where higher values mean greater apparent sensitivity.

In [ ]:
# Key PRISM response transformation used in the analysis:
# relative_abundance_percent = 100 * (2 ** prism_doxorubicin_lfc)
# doxorubicin_sensitivity_score = -1 * prism_doxorubicin_lfc

The small PRISM/DepMap sample size came from intersecting several data requirements, not from the total number of breast cancer cell lines in DepMap.

| attrition_step | count | note |
| --- | --- | --- |
| Total rows/models in DepMap Model.csv | 1719 | ID column: ModelID |
| Models labelled as breast cancer / breast lineage | 71 | Filtered using OncotreeLineage and OncotreePrimaryDisease |
| Breast cancer models with expression data | 71 | Expression table contains 1719 total model IDs |
| Breast cancer models with selected p53/DNA-damage gene expression columns | 71 | Genes found: 16 of 16 |
| Doxorubicin treatment rows found in PRISM treatment info | 1 | Matched by searching name for doxorubicin |
| Breast cancer models with any PRISM response data | 26 | Any row in primary-screen-replicate-collapsed-logfold-change.csv |
| Breast cancer models with non-missing PRISM doxorubicin LFC | 26 | Doxorubicin LFC aggregated across 1 available treatment column(s) |
| Final joined modelling table rows | 26 | Intersection of breast metadata, selected expression genes, and non-missing PRISM doxorubicin response |

![PRISM/DepMap attrition funnel](../figures/prism_depmap_attrition_funnel.png)

*Figure 1. Formation of the PRISM/DepMap modelling cohort.*

## Statistical Methods

TCGA-BRCA survival analyses used Cox proportional hazards models [5]. Where appropriate, age-adjusted Cox models were added to check whether the main score remained associated with survival after accounting for patient age. Median split Kaplan-Meier plots were used as visual summaries, but continuous Cox models were treated as the main survival analyses because dichotomising a continuous score loses information.

PRISM doxorubicin sensitivity was modelled using elastic-net regression for the 16 shared p53/DNA-damage genes [6]. Because the PRISM/DepMap breast cancer cohort contained only 26 cell lines, leave-one-out cross-validation was used as a simple sensitivity check for generalisation. Spearman and Pearson correlations were used for PRISM transfer analyses to summarise monotonic and linear associations.

All analyses were intentionally kept simple and interpretable. The aim was to support a transparent personalised medicine assignment rather than to optimise predictive performance.

## p53 ODE Model Explanation

The p53 ODE analysis used the given course template as a pre-specified mechanistic model of the DNA-damage response. In the model, **DDR** means DNA-damage response input: a simplified numerical signal representing increasing DNA-damage stimulation. The model is simulated across a range of DDR doses and returns p53 response summaries.

The main outputs used here were p53 S15 and p53 S46 simulated response scores. In simple biological terms, p53 S15 is treated as an early DNA-damage response activation marker, while p53 S46 is treated as a stronger stress/apoptosis-associated p53 response marker. The main pre-specified feature for Q1 and Q2 was `p53_S46_AUC_across_DDR`, the area under the simulated p53 S46 response curve across DDR inputs.

TCGA-BRCA and DepMap/CCLE are static baseline expression datasets, not p53 time-course experiments. Therefore, the ODE model was not learned from TCGA or PRISM time-course data. Instead, baseline expression of selected p53/DNA-damage genes was used to parameterise or score the pre-specified ODE model, and the resulting scores were tested against survival or drug response.

In [ ]:
# Conceptual p53 ODE scoring workflow
# For each sample or cell line:
#   1. use baseline p53/DNA-damage gene expression to set model inputs
#   2. simulate the given p53 ODE template across DDR doses
#   3. summarise p53 S15 and p53 S46 response curves
#   4. test the score against survival or PRISM doxorubicin sensitivity

## Results Q1: p53 ODE Model and TCGA-BRCA Survival

The main Q1 feature was `p53_S46_AUC_across_DDR`. In a continuous Cox model [5], higher p53 S46 AUC was modestly associated with lower overall-survival hazard in TCGA-BRCA: **HR 0.85 per SD, p=0.038**. The age-adjusted model was similar: **HR 0.84, p=0.027**. A median-split Kaplan-Meier comparison was weaker: **log-rank p=0.102**.

This suggests a modest prognostic association when the score is used continuously. The weaker dichotomised result is not surprising because median splits discard information. Importantly, this is a prognostic TCGA survival result, not direct evidence of doxorubicin response.

![Q1 TCGA survival by p53 S46 score](../figures/q1_p53_ode_tcga_km.png)

*Figure 2. Q1 TCGA-BRCA survival by p53 S46 AUC score.*

## Results Q2: p53 ODE Model and PRISM Doxorubicin Response

The same pre-specified p53 S46 AUC score did not predict PRISM doxorubicin sensitivity in the 26 breast cancer cell lines [3]. The association was essentially absent: **Spearman r=0.04, p=0.838** and **Pearson r=0.01, p=0.961**.

This negative result is important. It suggests that the ODE S46 score may capture survival-associated p53/DNA-damage biology in TCGA-BRCA, but it did not behave as a direct doxorubicin sensitivity predictor in this small PRISM breast cancer subset.

![Q2 p53 ODE score versus PRISM doxorubicin sensitivity](../figures/q2_p53_ode_score_vs_prism_doxorubicin_sensitivity.png)

*Figure 3. Q2 p53 ODE score versus PRISM doxorubicin sensitivity.*

## Results Q3: PRISM Doxorubicin Signature Transferred to TCGA-BRCA Survival

Q3 fitted a simple elastic-net model [6] using the 16 shared p53/DNA-damage genes to predict PRISM doxorubicin sensitivity from breast cancer cell-line expression. The apparent in-sample fit looked moderate: **R²=0.44, RMSE=1.10 and Spearman r=0.76**. However, leave-one-out cross-validation was poor: **R²=-0.83, RMSE=1.99 and Spearman r=-0.22**.

When the PRISM-derived signature was applied to TCGA-BRCA, it was not associated with overall survival: **HR 0.95, p=0.60**. The age-adjusted result was also null: **HR 0.94, p=0.55**, and the median-split Kaplan-Meier result was **p=0.82**.

The key interpretation is that apparent cell-line fit did not generalise, and the transferred score was not prognostic in TCGA-BRCA.

![Q3 leave-one-out cell-line prediction](../figures/q3_prism_signature_cellline_cv_predicted_vs_observed.png)

*Figure 4. Q3 leave-one-out PRISM prediction.*

In [ ]:
# Conceptual elastic-net signature score:
# Coefficients were learned in PRISM/DepMap cell lines.
# prism_signature_score = model_intercept + sum(gene_expression_z[gene] * coefficient[gene] for gene in shared_genes)

## Results Q4: TCGA Survival Signature Transferred to PRISM Doxorubicin Response

Q4 fitted the reverse transfer. A Cox model [5] based on the shared p53/DNA-damage genes produced a TCGA-derived survival-risk score. Inside TCGA-BRCA, this score was prognostic: **HR 2.72, p=9.5e-08**. The age-adjusted model remained associated with survival: **HR 2.42, p=3.4e-06**, and the median-split Kaplan-Meier result was **p=1.3e-06**.

However, when the TCGA-derived survival-risk score was applied to PRISM/DepMap cell lines, it did not correlate with doxorubicin sensitivity: **Spearman r=0.0065, p=0.975** and **Pearson r=-0.064, p=0.757**.

This means the patient survival-risk score was meaningful inside TCGA-BRCA, but it did not predict in vitro doxorubicin sensitivity.

![Q4 TCGA-derived risk score versus PRISM doxorubicin sensitivity](../figures/q4_tcga_score_vs_prism_doxorubicin_sensitivity.png)

*Figure 5. Q4 TCGA-derived risk score versus PRISM doxorubicin sensitivity.*

In [ ]:
# Conceptual TCGA Cox risk score:
# Coefficients were learned from TCGA-BRCA survival.
# tcga_survival_risk_score = sum(gene_expression_z[gene] * cox_coefficient[gene] for gene in shared_genes)

## Results Q5: Model Comparison

Q5 compares three related but different modelling approaches. The mechanistic ODE model simulates p53 DNA-damage response using the given course template. The PRISM elastic-net model is a data-driven cell-line doxorubicin sensitivity signature. The TCGA Cox model is a patient-derived survival-risk score.

| Analysis | Model type | Training domain | Test domain | Endpoint | Main result | Evidence |
| --- | --- | --- | --- | --- | --- | --- |
| p53 ODE S46 AUC -> TCGA-BRCA survival | mechanistic ODE | given p53 ODE course template | TCGA-BRCA patients | overall survival | HR 0.85, p=0.038; age-adjusted HR 0.84, p=0.027 | modest prognostic signal |
| p53 ODE S46 AUC -> PRISM doxorubicin sensitivity | mechanistic ODE | given p53 ODE course template | PRISM/DepMap breast cancer cell lines | doxorubicin sensitivity | Spearman r=0.04, p=0.838; Pearson r=0.01, p=0.961 | no drug-response association |
| elastic-net PRISM signature -> PRISM doxorubicin sensitivity, in-sample | elastic-net regression | PRISM/DepMap breast cancer cell lines | same PRISM/DepMap cell lines | doxorubicin sensitivity | R2=0.44, RMSE=1.10, Spearman r=0.76 | apparent in-sample fit only |
| elastic-net PRISM signature -> PRISM doxorubicin sensitivity, LOOCV | elastic-net regression | PRISM/DepMap breast cancer cell lines, leave-one-out folds | held-out PRISM/DepMap cell line in each fold | doxorubicin sensitivity | LOOCV R2=-0.83, RMSE=1.99, Spearman r=-0.22 | poor cross-validated generalisation |
| PRISM-derived signature -> TCGA-BRCA survival | elastic-net regression transfer | PRISM/DepMap breast cancer cell lines | TCGA-BRCA patients | overall survival | HR=0.95, p=0.60; age-adjusted HR=0.94, p=0.55; KM p=0.82 | no survival transfer |
| TCGA Cox gene score -> TCGA-BRCA survival | Cox regression | TCGA-BRCA patients | same TCGA-BRCA patients | overall survival | HR=2.72, p=9.5e-08; age-adjusted HR=2.42, p=3.4e-06; KM p=1.3e-06 | strong within-domain prognostic signal |
| TCGA Cox gene score -> PRISM doxorubicin sensitivity | Cox regression transfer | TCGA-BRCA patients | PRISM/DepMap breast cancer cell lines | doxorubicin sensitivity | Spearman r=0.0065, p=0.975; Pearson r=-0.064, p=0.757 | no drug-response transfer |

![Q5 model comparison summary](../figures/q5_model_comparison_summary.png)

*Figure 6. Q5 qualitative model comparison summary.*

The evidence summary score is qualitative and ordinal. It summarises whether each analysis showed useful evidence within-domain or after cross-domain transfer; it is not a directly comparable statistical effect size.

Across models, the strongest evidence appeared within-domain. The p53 ODE score and TCGA Cox gene score showed survival associations in TCGA-BRCA, but neither transferred to PRISM doxorubicin sensitivity. Conversely, the PRISM-derived doxorubicin signature showed some apparent in-sample fit but poor cross-validated performance and no association with TCGA-BRCA survival. Overall, these results suggest that patient survival prognosis and in vitro doxorubicin sensitivity are related but non-equivalent endpoints.

## Results Q6: LINCS feasibility assessment

Q6 was treated as a feasibility assessment because no usable LINCS doxorubicin perturbation file was available in the working repository [4]. No fabricated LINCS result was produced.

A complete LINCS analysis would require a doxorubicin perturbation expression signature with enough metadata to identify cell line, dose, time point, control condition and gene identifiers. With those files, the analysis could ask whether doxorubicin induces a p53/DNA-damage expression pattern involving genes such as CDKN1A, BAX, BBC3, GADD45A, MDM2, ATM, ATR, CHEK1 and CHEK2. That perturbation signature could then be compared qualitatively with the ODE p53 S15/S46 interpretation and with the TCGA/PRISM expression signatures.

Because those inputs were not present, Q6 supports the report as a transparent limitation rather than as a quantitative result.

## Discussion

Across Q1-Q6, the clearest pattern was that endpoint context mattered. TCGA-BRCA survival [1] and PRISM doxorubicin sensitivity [3] are biologically related but not equivalent. TCGA survival reflects tumour biology, treatment heterogeneity, patient age, stage, immune context, comorbidity and follow-up. PRISM measures in vitro relative abundance after drug treatment in cell lines, without tumour microenvironment or patient-level clinical complexity.

The p53 ODE score showed a modest survival association in TCGA-BRCA but no PRISM doxorubicin sensitivity association. This suggests that p53/DNA-damage response expression may carry some prognostic information without directly predicting doxorubicin response. Similarly, the Q3 and Q4 transfer analyses were negative: a PRISM-derived drug-response signature did not transfer to patient survival, and a TCGA-derived survival score did not transfer to cell-line doxorubicin sensitivity.

The Q3 in-sample result is a useful warning. The elastic-net model looked moderately predictive when assessed on the same 26 cell lines used for fitting, but leave-one-out cross-validation was poor. This illustrates how in-sample model fit can be misleading when sample size is small.

Negative transfer results are still informative. They show that a signature can be meaningful for one endpoint and unhelpful for another, which is a central issue in personalised medicine modelling.

## Limitations

The PRISM/DepMap analysis had only 26 breast cancer cell lines after intersecting lineage, expression, treatment metadata and non-missing doxorubicin LFC. This small sample size limits model stability and makes cross-validation estimates noisy.

TCGA overall survival is not direct doxorubicin response. TCGA-BRCA also includes treatment heterogeneity, and the available survival endpoint does not isolate anthracycline benefit. Age and stage were available, but the modelling remained deliberately simple for a transparent assignment analysis.

The gene set was restricted to 16 selected p53/DNA-damage genes. This keeps the assignment explainable but may miss other determinants of doxorubicin response, such as drug transport, proliferation, DNA repair beyond the selected genes and apoptosis context.

The p53 ODE model was adapted from the given template and was not fully re-calibrated for breast cancer. TCGA and DepMap/CCLE are static baseline expression snapshots, not p53 dynamic time-course measurements. There was no external validation dataset. LINCS analysis was limited because suitable doxorubicin perturbation files were not available in the repository.

## Conclusion

The analyses suggest that p53/DNA-damage response features can show modest prognostic signal in TCGA-BRCA, but neither mechanistic ODE features nor expression-based signatures transferred cleanly between patient survival and PRISM doxorubicin sensitivity. This highlights an important personalised medicine lesson: in vitro drug response and patient survival are related but non-equivalent endpoints.

## References

1. The Cancer Genome Atlas Network. Comprehensive molecular portraits of human breast tumours. *Nature*. 2012.
2. Ghandi M, Huang FW, Jané-Valbuena J, et al. Cancer Cell Line Encyclopedia study. *Nature*. 2019.
3. Corsello SM, Nagari RT, Spangler RD, et al. PRISM drug viability profiling study. *Nature Cancer*. 2020.
4. Subramanian A, Narayan R, Corsello SM, et al. Connectivity Map L1000 perturbation resource. *Cell*. 2017.
5. Cox DR. Regression models and life-tables. *Journal of the Royal Statistical Society: Series B*. 1972.
6. Zou H and Hastie T. Elastic net regression method. *Journal of the Royal Statistical Society: Series B*. 2005.